# ResNeSt — Split-Attention Networks

**Paper:** Zhang, H., Wu, C., Zhang, Z., Zhu, Y., Zhang, Z., Lin, H., ... & Smola, A. J. (2020).  
*ResNeSt: Split-Attention Networks.*  
arXiv:2004.08955. [[PDF]](https://arxiv.org/abs/2004.08955)

**Affiliation:** Amazon Web Services (AWS AI Labs)

---

## Overview

ResNeSt extends ResNet by introducing **Split-Attention**: a per-channel attention mechanism
over multiple feature map *splits* (branches).  Unlike SE-Net which performs global channel
attention, ResNeSt performs group-wise attention over radix splits.

Key difference from other attention mechanisms:
- **SE-Net**: channel attention, one global weight per channel
- **CBAM**: spatial + channel attention
- **ResNeSt**: attention across radix feature splits (like Selective Kernel generalised)

ResNeSt achieves state-of-the-art results on detection/segmentation tasks when used
as a backbone (Detectron2, ADE20K, Cityscapes).

## Architecture (ASCII)

### Split-Attention Conv2d (radix=2)
```
Input X [B, C_in, H, W]
    │
    ▼
  Conv(C_in → C_out × radix, groups=radix)   ← produce radix=2 branches
  BN + ReLU
  reshape → [B, radix=2, C_out, H, W]        U₁, U₂
    │
    ▼
  V = U₁ + U₂   (element-wise sum)           [B, C_out, H, W]
    │
    ▼
  s = GlobalAvgPool(V)                        [B, C_out, 1, 1]
  s = BN(s)
  s = FC(C_out → C_out//4) → ReLU            bottleneck
  s = FC(C_out//4 → C_out × radix)           split scores
  reshape → [B, radix=2, C_out, 1, 1]
  softmax over radix axis → a₁, a₂           attention weights
    │
    ▼
  Output = a₁ × U₁ + a₂ × U₂               [B, C_out, H, W]
```

### ResNeSt Bottleneck
```
input ─────────────────────────────── (+) ──→ ReLU
  ↓                                    ↑
  Conv 1×1 + BN + ReLU            (downsample?)
  ↓
  SplitAttentionConv2d(3×3)  ← replaces standard 3×3
  ↓
  ReLU
  ↓
  Conv 1×1 + BN
  ─────────────────────────────────────┘
```

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import numpy as np

BG   = '#0f1117'
FG   = '#e8eaf6'
BR1  = '#42a5f5'  # branch 1
BR2  = '#ef5350'  # branch 2
ATT  = '#ab47bc'  # attention
SUM  = '#ffa726'  # sum
OUT  = '#66bb6a'  # output

fig, axes = plt.subplots(1, 2, figsize=(15, 8), facecolor=BG)
fig.suptitle('ResNeSt: Split-Attention Mechanism', color=FG, fontsize=13, fontweight='bold')

# ── Left: Split-Attention diagram ─────────────────────────────────────────────
ax = axes[0]
ax.set_facecolor(BG); ax.set_xlim(0, 8); ax.set_ylim(0, 12); ax.axis('off')
ax.set_title('SplitAttentionConv2d (radix=2)', color=FG, fontsize=11)

def box(ax, x, y, w, h, text, color, alpha=0.7, fs=8.5):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1', fc=color, alpha=alpha, ec=FG, lw=1))
    ax.text(x+w/2, y+h/2, text, ha='center', va='center', color=FG, fontsize=fs)

def arr(ax, x, y1, y2, color=FG):
    ax.annotate('', xy=(x, y2), xytext=(x, y1), arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

# Input
box(ax, 2.0, 10.8, 4, 0.7, 'Input X [B, C_in, H, W]', FG, alpha=0.3)
arr(ax, 4.0, 10.8, 10.1)
box(ax, 1.5, 9.4, 5, 0.6, 'Conv(C_in → C_out×2, grouped) + BN + ReLU', ATT, 0.6)
arr(ax, 4.0, 9.4, 8.7)
# Split into 2 branches
ax.text(4.0, 8.5, 'reshape → 2 branches', ha='center', color=FG, fontsize=8)
arr(ax, 2.5, 8.7, 8.0)  # branch 1
arr(ax, 5.5, 8.7, 8.0)  # branch 2
box(ax, 0.5, 7.2, 3.5, 0.7, 'U₁ [B, C_out, H, W]', BR1)
box(ax, 4.0, 7.2, 3.5, 0.7, 'U₂ [B, C_out, H, W]', BR2)

# Sum for attention query
ax.annotate('', xy=(2.5, 6.5), xytext=(2.5, 7.2), arrowprops=dict(arrowstyle='->', color=SUM, lw=1.5))
ax.annotate('', xy=(5.5, 6.5), xytext=(5.5, 7.2), arrowprops=dict(arrowstyle='->', color=SUM, lw=1.5))
box(ax, 1.5, 5.8, 5, 0.6, 'V = U₁ + U₂   (element-wise sum)', SUM, 0.6)
arr(ax, 4.0, 5.8, 5.1)

# Attention branch
box(ax, 1.5, 4.4, 5, 0.6, 'GlobalAvgPool → BN → FC → ReLU → FC', ATT, 0.5)
arr(ax, 4.0, 4.4, 3.7)
box(ax, 1.5, 3.0, 5, 0.6, 'Softmax over radix → (a₁, a₂)', ATT)

# Final weighted sum
ax.annotate('', xy=(2.5, 2.5), xytext=(2.5, 3.0), arrowprops=dict(arrowstyle='->', color=BR1, lw=1.5))
ax.annotate('', xy=(5.5, 2.5), xytext=(5.5, 3.0), arrowprops=dict(arrowstyle='->', color=BR2, lw=1.5))
ax.text(1.5, 2.0, 'a₁ × U₁', ha='center', color=BR1, fontsize=9, fontweight='bold')
ax.text(4.5, 2.0, '+ a₂ × U₂', ha='center', color=BR2, fontsize=9, fontweight='bold')
arr(ax, 4.0, 1.7, 1.1)
box(ax, 1.5, 0.3, 5, 0.7, 'Output [B, C_out, H, W]', OUT)

# ── Right: Comparison with SE ──────────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor(BG); ax2.set_xlim(0, 6); ax2.set_ylim(-1, 12); ax2.axis('off')
ax2.set_title('Comparison: SE-Net vs ResNeSt Attention', color=FG, fontsize=11)

# SE-Net summary
box(ax2, 0.3, 9.5, 5.4, 1.2, 'SE-Net: Global Channel Attention\nPool → [B,C,1,1] → FC → Sigmoid → scale X\n(1 weight per channel, all spatial positions)', '#5c6bc0')
box(ax2, 0.3, 7.5, 5.4, 1.6, 'ResNeSt: Split Attention\nCreate radix branches → sum → attention over splits\n(radix attention weights per channel group)\nGeneralises Selective Kernel Networks (SKNet)', '#ef5350', 0.5)

# Accuracy table
table_data = [
    ['Model', 'Params', 'GFLOPs', 'ImageNet Top-1'],
    ['ResNet-50', '25M', '4.1', '76.2%'],
    ['SE-ResNet-50', '28M', '4.1', '77.6%'],
    ['ResNeSt-50', '27M', '5.4', '81.1%'],
    ['ResNeSt-101', '48M', '10.2', '82.8%'],
    ['ResNeSt-269', '110M', '77.9', '84.5%'],
]
for i, row in enumerate(table_data):
    y = 6.0 - i * 0.8
    for j, cell in enumerate(row):
        x = 0.3 + j * 1.35
        color = FG if i == 0 else ('#ab47bc' if 'ResNeSt' in cell else FG)
        ax2.text(x, y, cell, color=color, fontsize=8,
                 fontweight='bold' if i == 0 else 'normal')

ax2.axhline(y=5.65, xmin=0.05, xmax=0.95, color=FG, lw=0.5, alpha=0.5)
ax2.text(3.0, -0.3, 'ResNeSt achieves SOTA on detection & segmentation', 
         ha='center', color=OUT, fontsize=8.5, style='italic')

plt.tight_layout()
plt.savefig('resnest_diagram.png', dpi=120, bbox_inches='tight', facecolor=BG)
plt.show()

## Key Innovations & Mathematics

### Split-Attention Formulation

For cardinality $K=1$, radix $r=2$, input $X \in \mathbb{R}^{B \times C \times H \times W}$:

**Step 1 — Feature Splits:**
$$\hat{U}_k = \mathcal{F}_k(X), \quad k \in \{1, \ldots, r\}$$

where $\mathcal{F}_k$ is a 3×3 grouped convolution applied to split $k$.

**Step 2 — Aggregation:**
$$V = \sum_{k=1}^{r} \hat{U}_k \quad \in \mathbb{R}^{B \times C \times H \times W}$$

**Step 3 — Soft Attention (split scores):**
$$s = \text{GlobAvgPool}(V) \in \mathbb{R}^{B \times C}$$
$$a_k = \text{Softmax}(W_2 \cdot \delta(W_1 s))_k \in \mathbb{R}^{B \times C}$$

where $W_1 \in \mathbb{R}^{C/4 \times C}$, $W_2 \in \mathbb{R}^{rC \times C/4}$.

**Step 4 — Weighted Combination:**
$$Y_c = \sum_{k=1}^{r} a_k^c \cdot \hat{U}_k^c$$

When $r=1$: reduces to standard grouped convolution.  
When $K>1$: each group of channels has independent split-attention.

### Why Softmax over Sigmoid?

Using **softmax** over radix forces the attention to be a convex combination: $\sum_k a_k = 1$.  
This means the network must *choose* which split to emphasise — a competitive, sparse pressure.  
SE-Net uses sigmoid (independent per channel), which can amplify all channels simultaneously.

In [ ]:
# ── Code Walkthrough ──────────────────────────────────────────────────────────
import sys; sys.path.insert(0, '../../src')
import torch
from hale_core.nn.vision.cnn.resnest import SplitAttentionConv2d, ResNeStBottleneck

# 1. SplitAttentionConv2d — forward pass
sa = SplitAttentionConv2d(in_channels=64, out_channels=64, kernel_size=3, radix=2)
x  = torch.randn(2, 64, 28, 28)
y  = sa(x)
print('SplitAttentionConv2d(64→64, k=3, radix=2):')
print(f'  Input:  {tuple(x.shape)}')
print(f'  Output: {tuple(y.shape)}')

# 2. Attention weights are soft-max over radix, so they sum to 1
print('\n  Conv produces C×radix feature maps, then attention recombines them.')
print(f'  sa.conv output channels: {sa.conv.out_channels}  (= 64 × 2)')

# 3. ResNeSt Bottleneck
blk = ResNeStBottleneck(in_planes=256, planes=64)  # expands to 256
x2  = torch.randn(1, 256, 28, 28)
y2  = blk(x2)
print(f'\nResNeStBottleneck(256, 64): input={tuple(x2.shape)}, output={tuple(y2.shape)}')
print(f'  Params: {sum(p.numel() for p in blk.parameters()):,}')

In [ ]:
# ── Usage Example ─────────────────────────────────────────────────────────────
from hale_core.nn.vision import build_vision_backbone

for name in ['resnest50', 'resnest101']:
    b = build_vision_backbone(name)
    b.eval()
    x = torch.randn(1, 3, 224, 224)
    with torch.no_grad():
        out = b(x)
    print(f'{name}:')
    print(f'  pooled:       {tuple(out.pooled_output.shape)}')
    print(f'  feature_maps: {[tuple(f.shape) for f in out.feature_maps]}')
    print(f'  params:       {b.num_parameters():,}')

## References

| # | Citation |
|---|----------|
| [1] | Zhang, H., Wu, C., Zhang, Z., Zhu, Y., Zhang, Z., Lin, H., Sun, Y., He, T., Mueller, J., Manmatha, R., Li, M., & Smola, A. (2020). **ResNeSt: Split-Attention Networks.** arXiv:2004.08955. |
| [2] | Li, X., Wang, W., Hu, X., & Yang, J. (2019). **Selective Kernel Networks.** CVPR. arXiv:1903.06586. |
| [3] | Hu, J., Shen, L., & Sun, G. (2018). **Squeeze-and-Excitation Networks.** CVPR. arXiv:1709.01507. |
| [4] | He, K., Zhang, X., Ren, S., & Sun, J. (2016). **Deep Residual Learning for Image Recognition.** CVPR. arXiv:1512.03385. |
| [5] | Xie, S., Girshick, R., Dollár, P., Tu, Z., & He, K. (2017). **Aggregated Residual Transformations for Deep Neural Networks (ResNeXt).** CVPR. |

---
*Notebook: ResNeSt — HaleBlocks backbone family*  
*Architecture: `hale_core.nn.vision.cnn.resnest`*